# Topic 7 — Temperature and Sampling

Hands-on notebook using your local Qwen3-4B `llama-server` from Topic 6.

Covers: temperature, greedy decoding, top-k, top-p, repeated sampling, task-specific settings, raw HTTP request bodies, and practice exercises.

**Start first:** 
```

llama-server -hf Qwen/Qwen3-4B-GGUF:Q4_K_M -ngl 0 -c 4096 --host 127.0.0.1 --port 8080

```

In [1]:
import json, time, urllib.error, urllib.request
from pprint import pprint

BASE_URL = 'http://127.0.0.1:8080'
CHAT_URL = f'{BASE_URL}/v1/chat/completions'
MODEL = 'Qwen/Qwen3-4B-GGUF:Q4_K_M'

print(CHAT_URL)

http://127.0.0.1:8080/v1/chat/completions


## 1. Raw HTTP helper

Sampling parameters are JSON body fields, not HTTP headers. No LLM SDK is used.

In [2]:
def post_json(url, body, timeout=180):
    req = urllib.request.Request(url, data=json.dumps(body).encode(), headers={'Content-Type':'application/json'}, method='POST')
    start = time.perf_counter()
    try:
        with urllib.request.urlopen(req, timeout=timeout) as r:
            return {'status': r.status, 'headers': dict(r.headers), 'body': json.loads(r.read().decode()), 'elapsed': time.perf_counter()-start}
    except urllib.error.HTTPError as e:
        raw = e.read().decode(errors='replace')
        try: body = json.loads(raw)
        except json.JSONDecodeError: body = {'raw': raw}
        return {'status': e.code, 'headers': dict(e.headers), 'body': body, 'elapsed': time.perf_counter()-start}

def answer_of(result):
    if result['status'] != 200: return None
    choice = result['body'].get('choices',[{}])[0]
    msg = choice.get('message',{})
    return {'content': msg.get('content'), 'reasoning_content': msg.get('reasoning_content'), 'finish_reason': choice.get('finish_reason'), 'usage': result['body'].get('usage',{})}

## 2. Baseline request

Keep the prompt fixed while changing only sampling parameters.

In [3]:
PROMPT = 'Give three different names for a small plant shop. Return one name per line.'
body = {'model': MODEL, 'messages':[{'role':'user','content':PROMPT}], 'temperature':0.7, 'max_tokens':150}
print(json.dumps(body, indent=2))
r = post_json(CHAT_URL, body)
print('HTTP:', r['status'])
pprint(answer_of(r))

{
  "model": "Qwen/Qwen3-4B-GGUF:Q4_K_M",
  "messages": [
    {
      "role": "user",
      "content": "Give three different names for a small plant shop. Return one name per line."
    }
  ],
  "temperature": 0.7,
  "max_tokens": 150
}
HTTP: 200
{'content': '',
 'finish_reason': 'length',
 'reasoning_content': 'Okay, the user wants three different names for a small '
                      'plant shop. Let me think about what makes a good name '
                      'for such a place. It should be catchy, maybe have a '
                      'touch of nature or something related to plants. Also, '
                      "since it's a small shop, the name shouldn't be too "
                      'complicated or too generic.\n'
                      '\n'
                      'First, I need to consider keywords related to plants: '
                      'green, leaf, bloom, sprout, root, vine, bud, flower, '
                      'etc. Maybe combine those with words that suggest a cozy '

## 3. Temperature reshapes the distribution

For logits $z_i$: $p_i=e^{z_i/T}/\sum_j e^{z_j/T}$. Lower positive temperature sharpens the distribution; higher temperature flattens it. Temperature does not reorder logits.

In [4]:
import math

def softmax(xs):
    m=max(xs); ex=[math.exp(x-m) for x in xs]; s=sum(ex); return [x/s for x in ex]

def temp_probs(logits, T):
    return softmax([x/T for x in logits])

logits=[3.0,2.0,1.0,0.0]
for T in [0.5,1.0,2.0]:
    print(f'T={T}: {[round(x,4) for x in temp_probs(logits,T)]}')

T=0.5: [0.865, 0.1171, 0.0158, 0.0021]
T=1.0: [0.6439, 0.2369, 0.0871, 0.0321]
T=2.0: [0.4551, 0.276, 0.1674, 0.1015]


## 4. Temperature experiment

Run the same prompt at low, medium, and high temperature.

In [5]:
def run_temperature(T, prompt=PROMPT):
    return answer_of(post_json(CHAT_URL, {'model':MODEL,'messages':[{'role':'user','content':prompt}],'temperature':T,'max_tokens':150}))

temperature_results={}
for T in [0.0,0.2,0.7,1.0]:
    print(f'\n===== temperature={T} =====')
    temperature_results[T]=run_temperature(T)
    pprint(temperature_results[T])


===== temperature=0.0 =====
{'content': '',
 'finish_reason': 'length',
 'reasoning_content': 'Okay, the user wants three different names for a small '
                      'plant shop. Let me think about what makes a good name '
                      'for a plant shop. It should be catchy, maybe a bit '
                      'whimsical or nature-related. Words like "bloom," '
                      '"green," "leaf," "vitality," "nurture," "grow," '
                      '"flora," "herb," "botanical," "roots," "foliage," '
                      '"petals," "bloom," "greenhouse," "tender," "tender '
                      'care," "earth," "earthly," "leafy," "lush," "vibrant," '
                      '"fresh," "green," "greenery," "bloom," "bloom," '
                      '"bloom,"',
 'usage': {'completion_tokens': 150,
           'prompt_tokens': 24,
           'prompt_tokens_details': {'cached_tokens': 23},
           'total_tokens': 174}}

===== temperature=0.2 =====
{'content': '',
 

## 5. Temperature 0 and reproducibility

Greedy/temperature-zero decoding removes ordinary sampling randomness, but is not a universal reproducibility contract. Test your actual local setup instead of assuming.

In [6]:
def repeated_outputs(body, n=5):
    rows=[]
    for i in range(n):
        r=post_json(CHAT_URL,body); a=answer_of(r)
        rows.append(a['content'] if a else None)
    return rows

greedy={'model':MODEL,'messages':[{'role':'user','content':'Give one short definition of an API.'}],'temperature':0.0,'max_tokens':100}
for i,out in enumerate(repeated_outputs(greedy),1): print(i, out)
print('Unique outputs:', len(set(repeated_outputs(greedy))))

1 
2 
3 
4 
5 
Unique outputs: 1


## 6. Nonzero temperature

At nonzero temperature, an early sampled token can change later context and therefore later distributions.

In [7]:
creative={'model':MODEL,'messages':[{'role':'user','content':'Give one unusual but professional name for a plant shop.'}],'temperature':1.0,'max_tokens':80}
outs=repeated_outputs(creative)
for i,out in enumerate(outs,1): print(i,out)
print('Unique outputs:',len(set(outs)))

1 
2 
3 
4 
5 
Unique outputs: 1


## 7. Top-k

Top-k keeps only the `k` highest-scoring candidates before sampling and renormalizes the retained distribution. Its candidate-set size has a fixed maximum.

The exact request field/support is server/model dependent, so inspect your installed `llama-server --help` before relying on it.

In [8]:
top_k_body={'model':MODEL,'messages':[{'role':'user','content':'Write one creative sentence about a rainy city.'}],'temperature':0.8,'top_k':20,'max_tokens':100}
r=post_json(CHAT_URL,top_k_body)
print('HTTP:',r['status']); pprint(answer_of(r) if r['status']==200 else r['body'])

HTTP: 200
{'content': '',
 'finish_reason': 'length',
 'reasoning_content': 'Okay, the user wants a creative sentence about a rainy '
                      'city. Let me think. First, I need to visualize a rainy '
                      'city. Maybe use some vivid imagery. Words like "slick," '
                      '"glistening," "drenched" could work. What about the '
                      'streets? They might be shiny, maybe with water '
                      'reflecting the sky. Maybe include some elements like '
                      'traffic, people, or buildings.\n'
                      '\n'
                      'Hmm, how about personifying the rain? Like the city is '
                      'whispering or something. Or',
 'usage': {'completion_tokens': 100,
           'prompt_tokens': 17,
           'prompt_tokens_details': {'cached_tokens': 3},
           'total_tokens': 117}}


## 8. Top-p / nucleus sampling

Top-p keeps the smallest highest-probability candidate set whose cumulative probability reaches `p`. The number of retained candidates changes with the distribution.

In [9]:
for p in [0.5,0.8,0.95,1.0]:
    body={'model':MODEL,'messages':[{'role':'user','content':'Give one creative name for a coffee shop.'}],'temperature':0.8,'top_p':p,'max_tokens':100}
    r=post_json(CHAT_URL,body)
    print(f'\n===== top_p={p} =====')
    pprint(answer_of(r) if r['status']==200 else r['body'])


===== top_p=0.5 =====
{'content': '',
 'finish_reason': 'length',
 'reasoning_content': 'Okay, the user wants a creative name for a coffee shop. '
                      'Let me think about what makes a good name. It should be '
                      'catchy, maybe have a bit of a theme or wordplay. Coffee '
                      'shops often use words related to coffee, like "brew," '
                      '"cup," "roast," but maybe I can be more original.\n'
                      '\n'
                      'Hmm, maybe combining words or using a metaphor. Words '
                      'like "mocha," "latte," "espresso" are common, but I '
                      'need something',
 'usage': {'completion_tokens': 100,
           'prompt_tokens': 17,
           'prompt_tokens_details': {'cached_tokens': 3},
           'total_tokens': 117}}

===== top_p=0.8 =====
{'content': '',
 'finish_reason': 'length',
 'reasoning_content': 'Okay, the user wants a creative name for a coffee shop. '
    

## 9. Compare the controls

**Temperature:** reshapes relative probabilities across candidates.

**Top-k:** crops to a fixed maximum number of candidates.

**Top-p:** crops to a probability-mass-based candidate set whose size varies.

Avoid stacking many knobs until you can explain each one's contribution.

## 10. Task-specific starting settings

Starting points from the Topic 7 material:

- Ideation: temperature about 0.7–1; optionally top-p.
- Extraction/classification: temperature about 0–0.2; constrain schema separately and validate.
- Code generation: temperature about 0–0.3; test the result.

Sampling settings cannot replace validation, schemas, compilation, or tests.

In [10]:
tasks={
 'ideation':(0.8,'Suggest five names for a small plant shop.'),
 'extraction':(0.1,'Extract the invoice number from: Invoice number INV-2026-1042. Return only the number.'),
 'code_generation':(0.2,'Write a Java method that returns the maximum integer in an array.')
}
for task,(T,prompt) in tasks.items():
    r=post_json(CHAT_URL,{'model':MODEL,'messages':[{'role':'user','content':prompt}],'temperature':T,'max_tokens':200})
    a=answer_of(r)
    print(f'\n===== {task} | temperature={T} =====\n{a["content"] if a else r["body"]}')


===== ideation | temperature=0.8 =====


===== extraction | temperature=0.1 =====


===== code_generation | temperature=0.2 =====



## 11. Provider parameter map

According to the supplied Topic 7 material:

- OpenAI Chat Completions: `temperature` and `top_p` are top-level JSON body fields; `top_k` is not a standard Chat Completions sampling field.
- Anthropic Messages: `temperature`, `top_p`, and `top_k` are top-level JSON body fields, with documented constraints on combining controls.

Always check the current provider/model endpoint contract before copying a payload.

## 12. Practice

1. Pick settings for ideation, structured extraction, and code generation. Defend each choice.

2. Explain the difference between a flattened distribution and a cropped candidate set.

3. Give two reasons temperature zero may still not reproduce an answer exactly.

4. Point to the JSON body fields for OpenAI Chat Completions and Anthropic Messages and identify which exposes `top_k`.

Write your answers before reading the interview summary.

## 13. Interview-ready summary

> Temperature rescales logits before softmax, changing how concentrated the next-token distribution is. Top-k and top-p truncate the candidate set and renormalize it. I use more randomness when variation is useful, and less when consistency matters, while validating outputs independently. Temperature zero is usually greedy and near-deterministic on a fixed serving setup, but does not guarantee reproducibility across provider infrastructure or model changes.

## 14. Final hands-on challenge

Without copying the earlier cells:

1. Compare temperatures 0, 0.2, 0.7, and 1.0 on one fixed prompt.
2. Explain the differences using probability distributions.
3. Compare one top-k and one top-p configuration.
4. Run five temperature-zero requests and record what your setup actually does.
5. Run five nonzero-temperature requests and record the variation.
6. Choose settings for ideation, extraction, and code generation.
7. Inspect the raw JSON body and identify every sampling parameter.
8. Explain why sampling cannot replace validation.

Do not move on until you can explain temperature vs top-k vs top-p without relying on memorized definitions.